In [ ]:
from google.colab import drive

drive.mount('/content/drive')
%cd /content/drive/MyDrive
!ls
!unzip "/content/drive/MyDrive/images_dataset/Archive.zip" -d "/content/drive/MyDrive/images_dataset/"

# **Pre-Process**

In [ ]:
import os
import librosa
from os import walk
import numpy as np
import librosa.display
import matplotlib.pyplot as plt

def Wav_To_melSpect_imgs(name):
    n_fft = 1024
    hop_length = 256
    n_mels = 40
    f_min = 20
    f_max = 8000
    plotFile = "_melSpectrogram_Plots"
    if not os.path.exists(name + plotFile):
        os.makedirs(name + plotFile)
        print ("# directory creation: "+str(name + plotFile))
    _wavs = []
    for (_,_,filenames) in walk(name):
        _wavs.extend(filenames)
        break
    for _wav in _wavs:
        # read audio samples
        if(".wav" in _wav ):
            print (_wav)
            file = name+"/" + _wav
            signal, sample_rate = librosa.load(file, sr = 44100)
            #
            mel_spec = librosa.feature.melspectrogram(signal, n_fft=n_fft, hop_length=hop_length,
                                n_mels=n_mels, sr=sample_rate, power=1.0, fmin=f_min, fmax=f_max)
            mel_spec_db = librosa.amplitude_to_db(mel_spec, ref=np.max)
            librosa.display.specshow(mel_spec_db, sr=sample_rate, x_axis='time')
            # label the axes
            plt.ylabel("Amplitude")
            plt.xlabel("Time")
            plt.axis('off')
            plt.savefig(name + plotFile + "/" + _wav.split('.')[0] + '.png')
            # plt.show()
            plt.close('all')
#Wav_To_melSpect_imgs("ourdataset/train/normal")
#Wav_To_melSpect_imgs("ourdataset/train/abnormal")



In [ ]:
# Split data into 3 folds
from glob import glob
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import os


def moveTrain(data, k):
    for i in range(len(data)):
        source = data[i]
        destination = data[i].replace("train", "train_kfold"+str(k))
        #dest = shutil.move(source, destination)
        dest = shutil.copy(source, destination)

def moveTest(data, k):
    for i in range(len(data)):
        source = data[i]
        destination = data[i].replace("train", "test_kfold"+str(k))
        #dest = shutil.move(source, destination)
        dest = shutil.copy(source, destination)

# importing shutil module
import shutil
cross_number = 3
kf = KFold(cross_number, True, 10)
normal_Plots = np.array(glob('train/normal/*.png'))
abnormal_Plots = np.array(glob('train/abnormal/*.png'))


#normal
kf.get_n_splits(normal_Plots)
i=1
for normal_Plots_train_index, normal_Plots_test_index in kf.split(normal_Plots):
    normal_Plots_train = normal_Plots[normal_Plots_train_index]
    normal_Plots_test = normal_Plots[normal_Plots_test_index]


    if not os.path.exists("train_kfold"+str(i)) and not os.path.exists("test_kfold"+str(i)):
        os.mkdir("train_kfold"+str(i))
        os.mkdir("test_kfold"+str(i))

    if (not os.path.exists("train_kfold"+str(i)+"/normal") and not os.path.exists("test_kfold"+str(i)+"/normal")):
        os.mkdir("train_kfold"+str(i)+"/normal")
        os.mkdir("test_kfold"+str(i)+"/normal")

    moveTrain(normal_Plots_train, i)
    moveTest(normal_Plots_test, i)

    i+=1

#abnormal
kf.get_n_splits(abnormal_Plots)
i=1
for abnormal_Plots_train_index, abnormal_Plots_test_index in kf.split(abnormal_Plots):
    abnormal_Plots_train = abnormal_Plots[abnormal_Plots_train_index]
    abnormal_Plots_test = abnormal_Plots[abnormal_Plots_test_index]


    if not os.path.exists("train_kfold"+str(i)) and not os.path.exists("test_kfold"+str(i)):
        os.mkdir("train_kfold"+str(i))
        os.mkdir("test_kfold"+str(i))

    if (not os.path.exists("train_kfold"+str(i)+"/abnormal") and not os.path.exists("test_kfold"+str(i)+"/abnormal")):
        os.mkdir("train_kfold"+str(i)+"/abnormal")
        os.mkdir("test_kfold"+str(i)+"/abnormal")

    moveTrain(abnormal_Plots_train, i)
    moveTest(abnormal_Plots_test, i)

    i+=1

# **Deep Learning CNN process**

In [ ]:
!pip install scikit-learn
!pip install keras-tqdm==1.0.1
!pip install pycm==0.8.1

In [ ]:
from keras.models import Model
from keras.layers import Dense, GlobalAveragePooling2D, Dropout
from keras.applications.inception_v3 import InceptionV3, preprocess_input
import time
from keras.models import load_model
from keras.callbacks import ModelCheckpoint
from tqdm import tqdm
# keras imports
from keras.applications.vgg16 import VGG16, preprocess_input
from keras.applications.vgg19 import VGG19, preprocess_input
from keras.applications.xception import Xception, preprocess_input
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from keras.applications.inception_resnet_v2 import InceptionResNetV2, preprocess_input
from keras.applications.mobilenet import MobileNet, preprocess_input
from keras.applications.inception_v3 import InceptionV3, preprocess_input
from keras.applications.densenet import DenseNet121
from keras.applications.densenet import DenseNet169
from keras.applications.densenet import DenseNet201
from keras.applications.nasnet import NASNetLarge
from keras.applications.nasnet import NASNetMobile
from keras.applications.mobilenet_v2 import MobileNetV2

from keras.applications.resnet import ResNet101
from keras.applications.resnet import ResNet152
from keras.applications.resnet_v2 import ResNet50V2
from keras.applications.resnet_v2 import ResNet101V2
from keras.applications.resnet_v2 import ResNet152V2
# filter warnings
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)
from keras_tqdm import TQDMNotebookCallback
from keras.applications.vgg16 import VGG16, preprocess_input
from keras.applications.vgg19 import VGG19, preprocess_input
from keras.applications.xception import Xception, preprocess_input
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from keras.applications.inception_resnet_v2 import InceptionResNetV2, preprocess_input
from keras.applications.mobilenet import MobileNet, preprocess_input
from keras.applications.inception_v3 import InceptionV3, preprocess_input
from keras.preprocessing import image
from keras.models import Model
from keras.models import model_from_json
from keras.layers import Input
from keras.layers import Dense,GlobalAveragePooling2D
from keras.callbacks import ModelCheckpoint
from pycm import *
import numpy as np
from glob import glob
import matplotlib.pyplot as plt
from numpy import array
from sklearn.model_selection import KFold
from tensorflow.keras.utils import load_img
from tqdm import tqdm_notebook as tqdm
from keras_tqdm import TQDMNotebookCallback
from keras_tqdm import TQDMCallback
from tensorflow.keras.utils import img_to_array

from keras.models import load_model
from keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import SGD, Adam
from keras.layers import MaxPooling2D
import keras
from keras import optimizers
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report, confusion_matrix
from keras.layers import Flatten
from tensorflow.keras.layers import (
    BatchNormalization, SeparableConv2D, MaxPooling2D, Activation, Flatten, Dropout, Dense
)
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D
def plot_acc_loss(history, PLOT_NAME, Folder_Training_name):
    fig = plt.figure(figsize=(10,5))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'])
    plt.plot(history.history['val_accuracy'])
    plt.title('model accuracy')
    plt.ylabel('accuracy')
    plt.xlabel('epoch')
    plt.legend(['train', 'validation'], loc='upper left')

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title('model loss')
    plt.ylabel('loss')
    plt.xlabel('epoch')
    plt.legend(['train', 'test'], loc='upper right')
    plt.savefig("/content/drive/My Drive/"+str(Folder_Training_name)+"/output/"+str(PLOT_NAME))
    plt.show()
def getPrediction(l):
  res = []
  i=0
  while i<len(l):
    if(l[i][0]==True):
      res.append(1)
    else:
      res.append(0)
    i+=1
  return res
#---------------------------------------------------------------------- Main
all_deep_models = [VGG16, VGG19, MobileNet, InceptionV3, InceptionResNetV2, Xception, DenseNet121, DenseNet169, DenseNet201, NASNetMobile, MobileNetV2, ResNet50, ResNet101, ResNet152, ResNet50V2, ResNet101V2, ResNet152V2, NASNetLarge]
all_model_name_txt = ["VGG16", "VGG19", "MobileNet", "InceptionV3", "InceptionResNetV2", "Xception", "DenseNet121", "DenseNet169", "DenseNet201", "NASNetMobile", "MobileNetV2", "ResNet50", "ResNet101", "ResNet152", "ResNet50V2", "ResNet101V2", "ResNet152V2", "NASNetLarge"]



CLASSES = 2
WIDTH = 640
HEIGHT = 480
BATCH_SIZE = 20
EPOCHS = 30
learning_rate = 0.000001
all_Kfolds = ["_kfold1", "_kfold2", "_kfold3"]

all_folder_training_names = [
                             "images_dataset"
                             ]

for Folder_Training_name in all_folder_training_names:
  print ("---------------------------------------------------------------------------")
  print ("- Training Folder Name: "+str(Folder_Training_name))
  print ("---------------------------------------------------------------------------")
  i=0
  for deep_model in all_deep_models:
    model_name = deep_model
    model_name_txt = str(all_model_name_txt[i])
    print ("- Deep model: "+str(model_name_txt))
    for KFOLD_ in all_Kfolds:

      print ("-KFold: "+str(KFOLD_))


      TRAIN_DIR = "/content/drive/My Drive/"+str(Folder_Training_name)+"/train"+str(KFOLD_)+"/"
      TEST_DIR = "/content/drive/My Drive/"+str(Folder_Training_name)+"/test"+str(KFOLD_)+"/"
      FOLD_NAME = str(model_name_txt)+str(KFOLD_)+".txt"
      Train_TimeFOLD_NAME = "TrainingTime_"+str(model_name_txt)+str(KFOLD_)+".txt"
      PLOT_NAME = model_name_txt+"/"+str(model_name_txt)+str(KFOLD_)+".png"

      # setup model
      base_model = model_name(weights='imagenet', include_top=False, input_tensor=Input(shape=(480, 640, 3)))
      print ("# Base model architecture before update")
      base_model.summary()

      x=base_model.output
      x = Dropout(.3)(x)

      x=GlobalAveragePooling2D()(x)
      x=Dense(1024,activation='relu')(x) #we add dense layers so that the model can learn more complex functions and classify for better results.
      #x = Dropout(.3)(x)
      x=Dense(512,activation='relu')(x) #dense layer 2
      #x = Dropout(.3)(x)
      x=Dense(128,activation='relu')(x) #dense layer 3
        #Batch normalization is a technique for training very deep neural networks
        #that standardizes the inputs to a layer for each mini-batch. This has the effect of stabilizing the learning process
        #and dramatically reducing the number of training epochs required to train deep networks.
      x = BatchNormalization()(x)
      preds=Dense(1, activation='sigmoid')(x) #final layer with softmax activation
      model = Model(inputs=base_model.input, outputs=preds)
      # Say not to train first layer (ResNet) model as it is already trained
      model.layers[0].trainable = False
      i=0
      while i<len(model.layers):
        model.layers[i].trainable = False
        if(i>10):
          model.layers[i].trainable = True

        i=i+1

      #Since this is a binary classification, the final layer activation is sigmoid, and only consist of 1 cell.
      #Batch Normalization is added to reduce overfitting
      #The number of hidden layer (1024) is arbitrary, it can be increased or decreased later to find better result.
      #opt = optimizers.Adam(lr = 0.000001)

      adam = Adam(lr = learning_rate)
      model.compile(optimizer = adam,
                    loss='binary_crossentropy',
                    metrics=['accuracy'])

      print ("# Final model architecture after update")
      model.summary()
    #for bi class: loss = 'binary_crossentropy'
    #for muti class: loss='categorical_crossentropy'


      # data prep
      train_datagen = ImageDataGenerator()
      validation_datagen = ImageDataGenerator()
      print ("-- Load Training data: ")
      train_generator = train_datagen.flow_from_directory(
          TRAIN_DIR,
          target_size=(HEIGHT, WIDTH),
          batch_size=BATCH_SIZE,
          class_mode='binary')
      print ("-- Load Test data: ")
      validation_generator = validation_datagen.flow_from_directory(
          TEST_DIR,
          target_size=(HEIGHT, WIDTH),
          batch_size=BATCH_SIZE,
          shuffle=False,
          class_mode='binary')

      print ("------ validation_generator.class_indices ----------")
      print (validation_generator.class_indices)
      print ("----------------------------------------------------")
      STEP_SIZE_TRAIN = train_generator.n//train_generator.batch_size
      STEP_SIZE_VALID = validation_generator.n//validation_generator.batch_size


      STEPS_PER_EPOCH = STEP_SIZE_TRAIN
      VALIDATION_STEPS = STEP_SIZE_VALID

      #STEPS_PER_EPOCH = 5
      #VALIDATION_STEPS = 5
      print ("- EPOCHS: "+str(EPOCHS))
      print ("- BATCH_SIZE: "+str(BATCH_SIZE))
      #print ("- STEPS_PER_EPOCH: "+str(STEPS_PER_EPOCH))
      #print ("- VALIDATION_STEPS: "+str(VALIDATION_STEPS))
      #1 Epoch = 1 Forward pass + 1 Backward pass for ALL training samples.
      #Batch Size = Number of training samples in 1 Forward/1 Backward pass.
      # (With increase in Batch size, required memory space increases.)
      #Number of iterations = Number of passes i.e. 1 Pass = 1 Forward pass + 1 Backward pass
      # (Forward pass and Backward pass are not counted differently.)

      MODEL_FILE = "/content/drive/My Drive/"+str(Folder_Training_name)+"/output/"+str(model_name_txt)+"/"+KFOLD_+str(model_name_txt)+".model"
      start_time = time.time()

      checkpoint = ModelCheckpoint(MODEL_FILE,
                                  monitor='val_accuracy',
                                  verbose=1,
                                  mode='max',
                                  save_best_only=True,
                                  save_weights_only=False,
                                  period=1)

      history = model.fit(
          train_generator,
          epochs = EPOCHS,
          steps_per_epoch = 50,
          validation_steps = VALIDATION_STEPS,
          validation_data = validation_generator,
          callbacks=[checkpoint],
          verbose=1
          )
      #callbacks=[checkpoint]


      end_time = time.time()
      duration = end_time - start_time
      print("Total training time = "+str(duration))
      tt = "/content/drive/My Drive/"+str(Folder_Training_name)+"/output/"+str(model_name_txt)+"/"+Train_TimeFOLD_NAME
      f = open(tt, "w")
      f.write("\n"+str(duration))
      f.close()
      model.save(MODEL_FILE)
      plot_acc_loss(history, PLOT_NAME, Folder_Training_name)


      # evaluate loaded model on test data
      # load model
      model = load_model(MODEL_FILE)
      # summarize model.
      #model.summary()
      # evaluate the model

      training_score = model.evaluate_generator(train_generator,  verbose=1)
      print("- Training accuracy: "+str(training_score[1]*100))
      validation_score = model.evaluate_generator(validation_generator,  verbose=1)
      print("- Validation accuracy: "+str(validation_score[1]*100))

      #Confution Matrix and Classification Report
      probabilities = model.predict_generator(validation_generator)
      y_pred = probabilities > 0.5
      y_true = validation_generator.classes
      print('Confusion Matrix')
      print(confusion_matrix(y_true, y_pred))
      print('Classification Report')
      #target_names = ['Murmur', 'Normal', 'Extrahls']
      target_names = ['normal', 'abnormal']
      print(classification_report(validation_generator.classes, y_pred, target_names=target_names))
      print ("--- y_true")
      print (y_true)
      print ("--- y_pred")
      y_pred = np.array(getPrediction(y_pred))
      print (y_pred)
      results = "/content/drive/My Drive/"+str(Folder_Training_name)+"/output/"+str(model_name_txt)+"/"+FOLD_NAME
      f = open(results, "w")
      cm = ConfusionMatrix(actual_vector=y_true, predict_vector=y_pred) # Create CM From Data
      f.write("{}\n".format(cm))
      f.close()
    i+=1


---------------------------------------------------------------------------
- Training Folder Name: images_dataset
---------------------------------------------------------------------------
- Deep model: VGG16
-KFold: _kfold1
# Base model architecture before update
Model: "vgg16"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_3 (InputLayer)        [(None, 480, 640, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 480, 640, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 480, 640, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 240, 320, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 240, 320, 128)     738

/usr/local/lib/python3.7/dist-packages/keras/optimizers/optimizer_v2/adam.py:110: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super(Adam, self).__init__(name, **kwargs)


Found 2434 images belonging to 2 classes.


-- Load Test data: 
Found 1218 images belonging to 2 classes.
------ validation_generator.class_indices ----------
{'abnormal': 0, 'normal': 1}
----------------------------------------------------
- EPOCHS: 30
- BATCH_SIZE: 20
Epoch 1/30
50/50 [==============================] - ETA: 0s - loss: 0.6968 - accuracy: 0.5690
Epoch 1: val_accuracy improved from -inf to 0.67500, saving model to /content/drive/My Drive/images_dataset/output/VGG16/_kfold1VGG16.model


50/50 [==============================] - 151s 3s/step - loss: 0.6968 - accuracy: 0.5690 - val_loss: 0.6632 - val_accuracy: 0.6750
Epoch 2/30
50/50 [==============================] - ETA: 0s - loss: 0.6460 - accuracy: 0.6430
Epoch 2: val_accuracy improved from 0.67500 to 0.73917, saving model to /content/drive/My Drive/images_dataset/output/VGG16/_kfold1VGG16.model


50/50 [==============================] - 93s 2s/step - loss: 0.6460 - accuracy: 0.6430 - val_loss: 0.6236 - val_accuracy: 0.7392
Epoch 3/30
50/50 [==============================] - ETA: 0s - loss: 0.6247 - accuracy: 0.6630
Epoch 3: val_accuracy improved from 0.73917 to 0.77583, saving model to /content/drive/My Drive/images_dataset/output/VGG16/_kfold1VGG16.model


... (2 outputs left out)

50/50 [==============================] - 93s 2s/step - loss: 0.5611 - accuracy: 0.7370 - val_loss: 0.5009 - val_accuracy: 0.7958
Epoch 10/30
50/50 [==============================] - ETA: 0s - loss: 0.5159 - accuracy: 0.7540
Epoch 10: val_accuracy did not improve from 0.79583
50/50 [==============================] - 89s 2s/step - loss: 0.5159 - accuracy: 0.7540 - val_loss: 0.4938 - val_accuracy: 0.7800
Epoch 11/30
50/50 [==============================] - ETA: 0s - loss: 0.5171 - accuracy: 0.7750
Epoch 11: val_accuracy did not improve from 0.79583
50/50 [==============================] - 89s 2s/step - loss: 0.5171 - accuracy: 0.7750 - val_loss: 0.5146 - val_accuracy: 0.7867
Epoch 12/30
50/50 [==============================] - ETA: 0s - loss: 0.5086 - accuracy: 0.7720
Epoch 12: val_accuracy improved from 0.79583 to 0.80500, saving model to /content/drive/My Drive/images_dataset/output/VGG16/_kfold1VGG16.model


50/50 [==============================] - 92s 2s/step - loss: 0.5086 - accuracy: 0.7720 - val_loss: 0.4810 - val_accuracy: 0.8050
Epoch 13/30
50/50 [==============================] - ETA: 0s - loss: 0.5010 - accuracy: 0.7706
Epoch 13: val_accuracy did not improve from 0.80500
50/50 [==============================] - 89s 2s/step - loss: 0.5010 - accuracy: 0.7706 - val_loss: 0.5069 - val_accuracy: 0.7883
Epoch 14/30
50/50 [==============================] - ETA: 0s - loss: 0.5004 - accuracy: 0.7910
Epoch 14: val_accuracy did not improve from 0.80500
50/50 [==============================] - 89s 2s/step - loss: 0.5004 - accuracy: 0.7910 - val_loss: 0.4950 - val_accuracy: 0.7750
Epoch 15/30
50/50 [==============================] - ETA: 0s - loss: 0.4938 - accuracy: 0.7817
Epoch 15: val_accuracy improved from 0.80500 to 0.84167, saving model to /content/drive/My Drive/images_dataset/output/VGG16/_kfold1VGG16.model


50/50 [==============================] - 92s 2s/step - loss: 0.4938 - accuracy: 0.7817 - val_loss: 0.4055 - val_accuracy: 0.8417
Epoch 16/30
35/50 [====================>.........] - ETA: 14s - loss: 0.4904 - accuracy: 0.7886

KeyboardInterrupt: ignored